# M1 — 01 Cleaning / 车辆清洗

Run cells in order. Included scripts contain all processing functions. No missing-value imputation is written into the clean CSV.
逐步运行：函数完整代码在scripts目录。交接数据保留缺失。

In [1]:
from pathlib import Path
import sys,json,pandas as pd
ROOT = next(p for p in [Path.cwd(),Path.cwd().parent,Path.cwd()/'M1_EV_Deliverables'] if (p/'data/raw/source_profile.json').exists())
sys.path.insert(0,str(ROOT/'scripts'))
from source_loader import load_source
from m1_pipeline import clean_listings,missingness,mark_outliers,export_clean,compare_imputers
print('Project root:',ROOT)


Project root: /workspace/scratch/b784693b5fd7/M1_EV_Condition_Update


## 1. Load and select Electric rows / 加载及筛选

The full source is scanned if supplied in data/raw; otherwise the audited Electric snapshot is used. The workbook was originally scanned in full, not sampled.

In [2]:
raw,profile=load_source(ROOT)
print('Full source rows:',profile['total_rows'])
print('Electric rows:',len(raw))
print(raw.head(3).to_string(index=False))


Using included Electric subset snapshot; full workbook counts come from its audited source profile.
Full source rows: 762091
Electric rows: 16192
manufacturer             model  year  mileage   engine      transmission      drivetrain fuel_type  mpg            exterior_color      interior_color  accidents_or_damage  one_owner  personal_use_only              seller_name  seller_rating  driver_rating  driver_reviews_num  price_drop  price  source_excel_row
        Audi RS e-tron GT Base  2022   1502.0 Electric 2-Speed Automatic All-wheel Drive  Electric  NaN      Suzuka Gray Metallic Black / Express Red                  0.0        1.0                1.0 Elliott Bay Auto Brokers            4.7            NaN                   0      6000.0 132880              8499
        Audi RS e-tron GT Base  2022  11020.0 Electric 2-Speed Automatic All-wheel Drive  Electric  NaN Daytona Gray Pearl Effect           Arras Red                  0.0        1.0                1.0       Audi South Atlanta   

## 2–3. True-EV exclusions, duplicates, categories and derived fields

Assign listing_id once, after cohort definition. Preserve raw model names; use explicit model-family rules rather than only the first token. Eight ambiguous BEV-status rows are retained for review.

In [3]:
ev=clean_listings(raw,profile,ROOT)
print(pd.read_csv(ROOT/'data/diagnostics/data_quality_log.csv').to_string(index=False))
print(ev.model_line.value_counts().to_string())
print('BEV status review IDs:',ev.loc[ev.bev_status_review_flag.eq(1),'listing_id'].tolist())


Clean cohort: 15155 brands: 19 model families: 44
                               step  rows_before  rows_removed  rows_after                                                                                 reason
                 01 Electric filter       762091        745899       16192                            Keep fuel_type Electric; extraction from supplied cars.xlsx
              02 Non-BEV exclusions        16192           962       15230                      Explicit project non-BEV model-name rule; retain removal evidence
                03 Exact duplicates        15230            75       15155         Exact original-field duplicates; provenance column excluded from duplicate key
04 Category and feature preparation        15155             0       15155 Standardise drivetrains and model families; preserve gaps; preserve all price outliers
model_line
Tesla Model 3                     2170
Nissan Leaf                       1741
Tesla Model S                     1631
Ford Mustang

## 4a. Profile missing values / 查看缺失

MAR/MNAR are hypotheses, not identified mechanisms. Seller size is not observed. Preserve unknown history rather than coding it as a clean car.

In [4]:
profile_missing=missingness(ev,ROOT)
print(profile_missing[profile_missing.missing_count>0].to_string(index=False))


                   field  missing_count  missing_pct                       mechanism_hypothesis                                                                                                                            explanation
          price_drop_raw           5614    37.043880 Structural absence or MAR/MNAR; unverified Project assumes absent recorded reduction = zero. Original values and missing indicator retained; no causal or future-price-cut claim.
           seller_rating           3539    23.352029                        MAR/MNAR hypotheses       Rates differ across observed seller/brand groups; seller size is not measured, so small sellers cannot be asserted as the cause.
           driver_rating           1630    10.755526                        MAR/MNAR hypotheses                                                 Inspect driver_reviews_num and model-year; no reviews and new models may lack ratings.
          interior_color           1231     8.122732                        

## 5. Review extreme observations / 检查异常

IQR uses the whole model family for sparse year groups. Isolation Forest is a complementary descriptive screen. These flags use asking price and must NEVER be predictive inputs. The designated Toyota error stays in the master file; M5 excludes it only from fitting data.

In [5]:
ev,crosstab=mark_outliers(ev,ROOT)
print(crosstab.to_string())
print(ev.loc[ev.confirmed_error_flag.eq(1),['listing_id','model','price','error_review_status']].to_string(index=False))


IQR flags 248 Isolation flags 152 project-designated errors 1
iso_outlier      0    1
iqr_outlier            
0            14771  136
1              232   16
 listing_id        model  price                                 error_review_status
      13975 bZ4X Limited 333333 project_designated_error_not_independently_verified


## 4b. Compare median/mode with KNN / 比较填补方法

Fixed 80/20 IDs; five shuffled folds on development only. Scaling and imputers fit within each training fold. Binary history uses mode in BOTH methods. Project-designated errors are excluded only from fold fitting, not fold validation. Pilot uses log-price MAE as primary score, USD MAE as secondary. No held-out predictive score is computed.

In [6]:
summary,decision=compare_imputers(ev,ROOT)
print(summary.to_string(index=False))
print(json.dumps(decision,indent=2))


CV median 1 log-MAE 0.361607 seconds 0.0
CV median 2 log-MAE 0.364795 seconds 0.0
CV median 3 log-MAE 0.363634 seconds 0.0
CV median 4 log-MAE 0.362364 seconds 0.0
CV median 5 log-MAE 0.364301 seconds 0.0
CV knn 1 log-MAE 0.361407 seconds 1.3
CV knn 2 log-MAE 0.365029 seconds 1.3
CV knn 3 log-MAE 0.363538 seconds 1.2
CV knn 4 log-MAE 0.362293 seconds 1.2
CV knn 5 log-MAE 0.364592 seconds 1.2
method  mean_mae_log_price  sd_mae_log_price  mean_mae_usd  total_seconds  selected
   knn            0.363372          0.001524  16618.353742       6.250168     False
median            0.363340          0.001331  16620.281556       0.089165      True
{
  "selected_method": "median",
  "criterion": "Lower mean 5-fold validation MAE on log price; development 80% only",
  "development_rows": 12124,
  "held_out_test_rows": 3031,
  "test_performance_evaluated": false,
  "numeric_fields": [
    "age",
    "mileage",
    "seller_rating",
    "driver_rating"
  ],
  "binary_fields": [
    "accidents_or_dam

## 6. Export and handover / 输出交接

Keep gaps in the exported CSV. Numeric target and price-derived diagnostics are not features. Reuse the split IDs in M5; do not select a favourable split. The method paragraph, dictionary and report are prepared in the delivered package.

In [7]:
clean_path=export_clean(ev,ROOT)
print('Saved:',clean_path)
assert ev.listing_id.is_unique
assert len(ev)==15155
assert ev.seller_rating.isna().any()
assert ev.confirmed_error_flag.sum()==1
print('Rows:',len(ev),'columns:',len(ev.columns),'unique IDs:',ev.listing_id.nunique())
print('Missing values preserved; statistical outliers retained.')


Saved: /workspace/scratch/b784693b5fd7/M1_EV_Condition_Update/data/clean/ev_listings_clean_v2.csv
Rows: 15155 columns: 48 unique IDs: 15155
Missing values preserved; statistical outliers retained.


### Additional condition features (teacher feedback, 15 October)

We added seven transparent condition features: a low-mileage-for-age flag, a near-new flag, a clean-history flag, an equal-weight condition score, age and mileage bands, and a cumulative-use battery-wear proxy. Low mileage means annualised mileage below 7,500 miles, using a minimum age denominator of 0.5 years. Near-new means approximate age no greater than one year and mileage below 5,000 miles. Clean history means an explicit record of no accident or damage; missing history remains unknown. The condition score sums the three flags (0–3) and remains missing if any component is unknown. These definitions and cutoffs are project assumptions, not verified mechanical standards.

Age bands are 0–1, 2–3, 4–5 and 6+ years. Mileage bands are below 10,000, 10,000–below 30,000, 30,000–below 60,000 and 60,000+ miles. Battery-wear proxy is mileage divided by 10,000, representing cumulative use exposure only. The data contain no measured battery state of health, charging history or battery replacement history. This proxy cannot establish battery degradation, and is a linear rescaling of an existing predictor. Its added predictive benefit may therefore be zero. The old mileage-below-100 flag is preserved as very_low_mileage_flag.

All new features use fixed row-level rules and do not use prices, target statistics or full-sample quantiles. M5 should compare the baseline with the baseline plus these features on identical development-set CV folds, using the same model and tuning budget. Imputation and encoding must be fitted inside each training fold. Selection should use CV only; evaluate the held-out test after selection. Report log-scale MAE and dollar-scale MAE/RMSE/R² with units clearly distinguished. Predictive benefit has not yet been tested by M1. Hand-built scores are candidates, not validated vehicle inspections.


In [8]:
from condition_features import CONDITION_FEATURES
print(ev[CONDITION_FEATURES].head().to_string(index=False))
print(ev[CONDITION_FEATURES].isna().sum().to_string())

 low_mileage_for_age_flag  near_new_flag  clean_history_flag  condition_score  battery_wear_proxy age_band mileage_band
                      1.0            1.0                 1.0              3.0              0.1502      0-1       0-<10k
                      0.0            0.0                 1.0              1.0              1.1020      0-1     10k-<30k
                      1.0            0.0                 1.0              2.0              0.6741      0-1       0-<10k
                      0.0            0.0                 1.0              1.0              0.7822      0-1       0-<10k
                      1.0            1.0                 1.0              3.0              0.2319      0-1       0-<10k
low_mileage_for_age_flag      6
near_new_flag                 4
clean_history_flag          811
condition_score             817
battery_wear_proxy            6
age_band                      0
mileage_band                  6
